# Import Required Libraries

This section imports all necessary libraries for computer vision, OCR, deep learning model loading, and file handling.


In [ ]:
import cv2
import easyocr
import numpy as np
from ultralytics import YOLO
import os
import glob 
import torch
import torchvision
from torchvision.models.detection import fasterrcnn_resnet50_fpn
from torchvision.models.detection import ssd300_vgg16
try:
    from google.colab.patches import cv2_imshow
except ImportError:
    cv2_imshow = cv2.imshow


## Initialize OCR and Load YOLO Model

Here we initialize EasyOCR and load the trained YOLOv8 model.


In [ ]:
    # 1. Initialize the OCR Reader (use gpu=True for Colab's GPU)
print("Loading EasyOCR reader...")
reader = easyocr.Reader(['en'], gpu=True) 

    # 2. Load Your Custom YOLOv8 Model (from Drive)
print("Loading custom YOLOv8 model...")
model_path = r'C:/Users/faris/Documents/GitHub/AI-Project/models/best.pt'
model = YOLO(model_path)


In [ ]:
def process_video(video_path, output_video_path=None, skip_frames=1, ocr_every_n_frames=5):
    """
    Process a video file with YOLOv8 and OCR on number plates.
    
    Args:
        video_path (str): Path to input video file.
        output_video_path (str): Path to save annotated video (optional).
        skip_frames (int): Process every Nth frame (1 = all frames, 2 = every other frame).
        ocr_every_n_frames (int): Run OCR on detected plates only every Nth frame (to speed up).
    """
    # Open video
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        print(f"Error: Cannot open video {video_path}")
        return

    # Get video properties
    fps = int(cap.get(cv2.CAP_PROP_FPS))
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

    # Setup video writer if output path provided
    out = None
    if output_video_path:
        fourcc = cv2.VideoWriter_fourcc(*'mp4v')  # or 'XVID' for AVI
        out = cv2.VideoWriter(output_video_path, fourcc, fps // skip_frames, (width, height))

    print(f"Video info: {width}x{height}, {fps} fps, {total_frames} frames")
    print(f"Processing every {skip_frames} frame(s)...")

    frame_count = 0
    processed_count = 0

    while True:
        ret, frame = cap.read()
        if not ret:
            break

        frame_count += 1

        # Skip frames if needed
        if frame_count % skip_frames != 0:
            continue

        processed_count += 1

        # Run YOLO detection
        results = model(frame)  # model is already loaded globally
        result = results[0]

        # Get class names
        class_names = model.names
        names_dict = {v: k for k, v in class_names.items()}
        plate_class_id = names_dict.get('Plate')

        # If plate class exists, optionally run OCR
        if plate_class_id is not None and processed_count % ocr_every_n_frames == 0:
            # Loop through detected plates
            for box in result.boxes:
                if int(box.cls) == plate_class_id:
                    # Crop plate region
                    coords = box.xyxy[0].cpu().numpy().astype(int)
                    x1, y1, x2, y2 = coords
                    cropped_plate = frame[y1:y2, x1:x2]
                    
                    # Preprocess for OCR (optional but recommended)
                    gray_plate = cv2.cvtColor(cropped_plate, cv2.COLOR_BGR2GRAY)
                    # Optional: resize, denoise, threshold here
                    
                    # Run OCR
                    ocr_results = reader.readtext(gray_plate)
                    if ocr_results:
                        plate_text = ocr_results[0][1]
                        # Draw text near bounding box
                        cv2.putText(frame, plate_text, (x1, y1-10), 
                                    cv2.FONT_HERSHEY_SIMPLEX, 0.9, (0,255,0), 2)

        # Draw YOLO bounding boxes (result.plot() draws boxes and labels)
        annotated_frame = result.plot()

        # Display live (optional) – press 'q' to quit early
        cv2.imshow('Video Processing', annotated_frame)
        if cv2.waitKey(1) & 0xFF == ord('q'):
            break

        # Write frame to output video
        if out:
            out.write(annotated_frame)

        # Progress indicator
        if processed_count % 30 == 0:
            print(f"Processed {processed_count} frames...")

    # Cleanup
    cap.release()
    if out:
        out.release()
    cv2.destroyAllWindows()
    print(f"Done! Processed {processed_count} frames.")

## Define Input and Output Folders

Set image source folder and results destination folder.


In [ ]:
    # 3. Define Input and Output Folders
    #
input_folder = 'C:/Users/faris/Documents/GitHub/AI-Project/results'
output_folder = 'C:/Users/faris/Documents/GitHub/AI-Project/Project_Results1'

    # Create the output folder if it doesn't exist
os.makedirs(output_folder, exist_ok=True)
print(f"Results will be saved in: {output_folder}")


## Load Images from Folder

Collect all JPG and PNG images from the input directory.


In [ ]:
# Get a list of all .jpg, .png images in the input folder
image_files = glob.glob(os.path.join(input_folder, '*.jpg'))
image_files.extend(glob.glob(os.path.join(input_folder, '*.png')))

if not image_files:
    print(f"--- ERROR: No images found in {input_folder} ---")
    # Instead of return, you might want to exit or just skip processing
    import sys
    sys.exit("No images to process.")

print(f"Found {len(image_files)} images to process...")

# Define Main Function

This function controls the full workflow:
- Load OCR
- Load YOLO model
- Process images
- Detect plates
- Apply OCR
- Save results

## Process Each Image

Loop through each image, detect plates, apply OCR, and save results.


In [ ]:
def main():
    # 1. Initialize the OCR Reader (use gpu=True for Colab's GPU)
    print("Loading EasyOCR reader...")
    reader = easyocr.Reader(['en'], gpu=True) 

    # 2. Load Your Custom YOLOv8 Model (from Drive)
    print("Loading custom YOLOv8 model...")
    model_path = r'C:/Users/faris/Documents/GitHub/AI-Project/models/best.pt'
    model = YOLO(model_path)

    # 3. Define Input and Output Folders
    input_folder = 'C:/Users/faris/Documents/GitHub/AI-Project/results'
    output_folder = 'C:/Users/faris/Documents/GitHub/AI-Project/Project_Results1'

    os.makedirs(output_folder, exist_ok=True)
    print(f"Results will be saved in: {output_folder}")

    # Get a list of all .jpg, .png images in the input folder
    image_files = glob.glob(os.path.join(input_folder, '*.jpg'))
    image_files.extend(glob.glob(os.path.join(input_folder, '*.png')))

    if not image_files:
        print(f"--- ERROR: No images found in {input_folder} ---")
        return

    print(f"Found {len(image_files)} images to process...")

    # --- START THE LOOP ---
    for image_path in image_files:
        print(f"\n--- Processing: {os.path.basename(image_path)} ---")
        
        img = cv2.imread(image_path)
        if img is None:
            print(f"Could not read image: {image_path}")
            continue

        # 4. Run YOLOv8 Detection
        results = model(img)
        result = results[0]
        # ...existing code...
        annotated_img = result.plot()

        output_path = os.path.join(
        output_folder,
        f"result_{os.path.basename(image_path)}"
       ) 

        cv2.imwrite(output_path, annotated_img)

        print(f"Saved result to: {output_path}")

# Run the main function
main()

In [ ]:

video_file = r'C:/Users/faris/Documents/GitHub/AI-Project/test_video/20615417-uhd_3840_2160_30fps.mp4'   
output_file = r'C:/Users/faris/Documents/GitHub/AI-Project/annotated_output/20615417-uhd_3840_2160_30fps.mp4'

process_video(video_file, output_video_path=output_file, skip_frames=1, ocr_every_n_frames=3)